# Análise de Dados
---
Respostas às perguntas de negócio.

### P1. Como evoluíram os preços unitários registrados no BPS entre 2024 e 2026? 

In [0]:
SELECT
    CAST(ano_compra AS STRING) AS ano_compra, -- Transformação do ano para string para facilitar a visualização no gráfico
    COUNT(*) AS qtd_registros,
    COUNT(DISTINCT sk_produto) AS qtd_catmats,
    ROUND(AVG(vl_preco_unitario), 2) AS preco_medio,
    ROUND(PERCENTILE_APPROX(vl_preco_unitario, 0.50, 10000), 2) AS preco_mediano,
    ROUND(PERCENTILE_APPROX(vl_preco_unitario, 0.25, 10000), 2) AS q1,
    ROUND(PERCENTILE_APPROX(vl_preco_unitario, 0.75, 10000), 2) AS q3
FROM mvp_puc_engenhariadedados.gold.fato_compra
WHERE vl_preco_unitario IS NOT NULL
  AND vl_preco_unitario > 0
GROUP BY ano_compra
ORDER BY ano_compra;

Databricks visualization. Run in Databricks to view.

### P2. Quais itens (CATMATs) apresentam maior dispersão de preços no período? 

Para identificação dos CATMATs com maior dispersão de preços, foram utilizadas duas medidas complementares: o Intervalo Interquartil (IQR), que representa a dispersão entre o primeiro e o terceiro quartis, e o Coeficiente de Variação (CV), que relaciona o desvio-padrão à média.   

In [0]:
SELECT
    co_catmat,
    ds_item,
    COUNT(*) AS qtd_compras,
    ROUND(MIN(vl_preco_unitario), 2) AS preco_minimo,
    ROUND(PERCENTILE_APPROX(vl_preco_unitario, 0.25, 10000), 2) AS q1,
    ROUND(PERCENTILE_APPROX(vl_preco_unitario, 0.50, 10000), 2) AS mediana,
    ROUND(PERCENTILE_APPROX(vl_preco_unitario, 0.75, 10000), 2) AS q3,
    ROUND(MAX(vl_preco_unitario), 2) AS preco_maximo,
    ROUND(PERCENTILE_APPROX(vl_preco_unitario, 0.75, 10000)- PERCENTILE_APPROX(vl_preco_unitario, 0.25, 10000), 2) AS iqr
FROM mvp_puc_engenhariadedados.gold.vw_registros_elegiveis
GROUP BY co_catmat, ds_item
ORDER BY iqr DESC
LIMIT 20;

In [0]:
SELECT
    co_catmat,
    ds_item,
    COUNT(*) AS qtd_compras,
    ROUND(AVG(vl_preco_unitario), 2) AS preco_medio,
    ROUND(STDDEV(vl_preco_unitario), 2) AS desvio_padrao,
    ROUND(
        100.0 * STDDEV(vl_preco_unitario)
        / NULLIF(AVG(vl_preco_unitario), 0),
        2
    ) AS cv_percentual,
    ROUND(MIN(vl_preco_unitario), 2) AS preco_minimo,
    ROUND(PERCENTILE_APPROX(vl_preco_unitario, 0.25, 10000), 2) AS q1,
    ROUND(PERCENTILE_APPROX(vl_preco_unitario, 0.50, 10000), 2) AS mediana,
    ROUND(PERCENTILE_APPROX(vl_preco_unitario, 0.75, 10000), 2) AS q3,
    ROUND(MAX(vl_preco_unitario), 2) AS preco_maximo,
    ROUND(
        PERCENTILE_APPROX(vl_preco_unitario, 0.75, 10000)
        - PERCENTILE_APPROX(vl_preco_unitario, 0.25, 10000),
        2
    ) AS iqr
FROM mvp_puc_engenhariadedados.gold.vw_registros_elegiveis
GROUP BY co_catmat, ds_item
ORDER BY cv_percentual DESC
LIMIT 20;

### P3. Quais compras apresentam preços potencialmente atípicos em relação às demais compras do mesmo CATMAT? 

In [0]:
SELECT
    -- Total de registros potencialmente atípicos
    COUNT(*) AS qtd_potencialmente_atipicos,

    -- Potencialmente atípicos abaixo do limite inferior
    SUM(
        CASE
            WHEN tipo_atipicidade = 'Abaixo do limite inferior'
            THEN 1
            ELSE 0
        END
    ) AS qtd_abaixo_limite,

    -- Potencialmente atípicos acima do limite superior
    SUM(
        CASE
            WHEN tipo_atipicidade = 'Acima do limite superior'
            THEN 1
            ELSE 0
        END
    ) AS qtd_acima_limite,

    -- Percentual abaixo do limite inferior
    ROUND(
        100.0 * SUM(
            CASE
                WHEN tipo_atipicidade = 'Abaixo do limite inferior'
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS percentual_abaixo_limite,

    -- Percentual acima do limite superior
    ROUND(
        100.0 * SUM(
            CASE
                WHEN tipo_atipicidade = 'Acima do limite superior'
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS percentual_acima_limite
FROM mvp_puc_engenhariadedados.gold.vw_potenciais_outliers;

In [0]:
SELECT
    co_seq_bps,
    ano_compra,
    data_completa,
    co_catmat,
    ds_item,
    ROUND(vl_preco_unitario, 2) AS vl_preco_unitario,
    ROUND(limite_inferior, 2) AS limite_inferior,
    ROUND(limite_superior, 2) AS limite_superior,
    CASE
        WHEN tipo_atipicidade = 'Abaixo do limite inferior'
            THEN ROUND(limite_inferior - vl_preco_unitario, 2)
        WHEN tipo_atipicidade = 'Acima do limite superior'
            THEN ROUND(vl_preco_unitario - limite_superior, 2)
    END AS distancia_do_limite,
    tipo_atipicidade,
    tp_compra,
    modalidade,
    sg_uf,
    no_municipio,
    no_instituicao,
    no_fornecedor
FROM mvp_puc_engenhariadedados.gold.vw_potenciais_outliers
ORDER BY distancia_do_limite DESC;

-- NOTA: É importante observar que a tabela output a seguir apresenta 6661+ linhas, ao invés de 7807, pois o resultado foi truncado devido à limitação de tamanho pela visualização do Databricks Free Edition. Entretanto, ao realizar o download do resultado em formato arquivo CSV foi possível observar o resultado completo e conforme o esperado.

### P4a. Como os registros potencialmente atípicos em relação às demais compras do mesmo CATMAT se distribuem ao longo do tempo, por ano?

In [0]:
SELECT
    CAST(e.ano_compra AS STRING) AS ano_compra, -- Transformação do ano para string para facilitar a visualização no gráfico

    -- Total de registros elegíveis no ano
    COUNT(*) AS qtd_registros_elegiveis,

    -- Total de registros potencialmente atípicos
    COUNT(o.co_seq_bps) AS qtd_potencialmente_atipicos,

    -- Potencialmente atípicos abaixo do limite inferior
    SUM(
        CASE
            WHEN o.tipo_atipicidade = 'Abaixo do limite inferior'
            THEN 1
            ELSE 0
        END
    ) AS qtd_abaixo_limite,

    -- Potencialmente atípicos acima do limite superior
    SUM(
        CASE
            WHEN o.tipo_atipicidade = 'Acima do limite superior'
            THEN 1
            ELSE 0
        END
    ) AS qtd_acima_limite,

    -- Percentual total de registros potencialmente atípicos
    ROUND(
        100.0 * COUNT(o.co_seq_bps) / COUNT(*),
        2
    ) AS percentual_potencialmente_atipicos,

    -- Percentual abaixo do limite inferior
    ROUND(
        100.0 * SUM(
            CASE
                WHEN o.tipo_atipicidade = 'Abaixo do limite inferior'
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS percentual_abaixo_limite,

    -- Percentual acima do limite superior
    ROUND(
        100.0 * SUM(
            CASE
                WHEN o.tipo_atipicidade = 'Acima do limite superior'
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS percentual_acima_limite

FROM mvp_puc_engenhariadedados.gold.vw_registros_elegiveis e
LEFT JOIN mvp_puc_engenhariadedados.gold.vw_potenciais_outliers o
    ON e.co_seq_bps = o.co_seq_bps
GROUP BY e.ano_compra
ORDER BY e.ano_compra;

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

### P4b. Como os registros potencialmente atípicos em relação às demais compras do mesmo CATMAT se distribuem por tipo de compra?

In [0]:
SELECT
    e.tp_compra,

    -- Total de registros elegíveis por tipo de compra
    COUNT(*) AS qtd_registros_elegiveis,

    -- Total de registros potencialmente atípicos
    COUNT(o.co_seq_bps) AS qtd_potencialmente_atipicos,

    -- Potencialmente atípicos abaixo do limite inferior
    SUM(
        CASE
            WHEN o.tipo_atipicidade = 'Abaixo do limite inferior'
            THEN 1
            ELSE 0
        END
    ) AS qtd_abaixo_limite,

    -- Potencialmente atípicos acima do limite superior
    SUM(
        CASE
            WHEN o.tipo_atipicidade = 'Acima do limite superior'
            THEN 1
            ELSE 0
        END
    ) AS qtd_acima_limite,

    -- Percentual total de registros potencialmente atípicos
    ROUND(
        100.0 * COUNT(o.co_seq_bps) / COUNT(*),
        2
    ) AS percentual_potencialmente_atipicos,

    -- Percentual abaixo do limite inferior
    ROUND(
        100.0 * SUM(
            CASE
                WHEN o.tipo_atipicidade = 'Abaixo do limite inferior'
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS percentual_abaixo_limite,

    -- Percentual acima do limite superior
    ROUND(
        100.0 * SUM(
            CASE
                WHEN o.tipo_atipicidade = 'Acima do limite superior'
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS percentual_acima_limite

FROM mvp_puc_engenhariadedados.gold.vw_registros_elegiveis e
LEFT JOIN mvp_puc_engenhariadedados.gold.vw_potenciais_outliers o
    ON e.co_seq_bps = o.co_seq_bps
GROUP BY e.tp_compra
ORDER BY percentual_potencialmente_atipicos DESC;

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

### 4c. Como os registros potencialmente atípicos em relação às demais compras do mesmo CATMAT se distribuem por modalidade de compra?

In [0]:
SELECT
    e.modalidade,

    -- Total de registros elegíveis por modalidade
    COUNT(*) AS qtd_registros_elegiveis,

    -- Representatividade da modalidade quando comparada com o total de registros elegíveis
    CASE
        WHEN COUNT(*) < 100
            THEN 'Baixa'
        ELSE 'Adequada'
    END AS representatividade,

    -- Total de registros potencialmente atípicos
    COUNT(o.co_seq_bps) AS qtd_potencialmente_atipicos,

    -- Potencialmente atípicos abaixo do limite inferior
    SUM(
        CASE
            WHEN o.tipo_atipicidade = 'Abaixo do limite inferior'
            THEN 1
            ELSE 0
        END
    ) AS qtd_abaixo_limite,

    -- Potencialmente atípicos acima do limite superior
    SUM(
        CASE
            WHEN o.tipo_atipicidade = 'Acima do limite superior'
            THEN 1
            ELSE 0
        END
    ) AS qtd_acima_limite,

    -- Percentual total de registros potencialmente atípicos
    ROUND(
        100.0 * COUNT(o.co_seq_bps) / COUNT(*),
        2
    ) AS percentual_potencialmente_atipicos,

    -- Percentual abaixo do limite inferior
    ROUND(
        100.0 * SUM(
            CASE
                WHEN o.tipo_atipicidade = 'Abaixo do limite inferior'
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS percentual_abaixo_limite,

    -- Percentual acima do limite superior
    ROUND(
        100.0 * SUM(
            CASE
                WHEN o.tipo_atipicidade = 'Acima do limite superior'
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS percentual_acima_limite

FROM mvp_puc_engenhariadedados.gold.vw_registros_elegiveis e
LEFT JOIN mvp_puc_engenhariadedados.gold.vw_potenciais_outliers o
    ON e.co_seq_bps = o.co_seq_bps
GROUP BY e.modalidade
ORDER BY percentual_potencialmente_atipicos DESC;

### P4d. Como os registros potencialmente atípicos em relação às demais compras do mesmo CATMAT se distribuem por fornecedor, entre aqueles com volume financeiro relevante?

In [0]:
SELECT
    e.cnpj_fornecedor,
    e.no_fornecedor,

    -- Total de registros elegíveis por fornecedor
    COUNT(*) AS qtd_registros_elegiveis,

    -- Representatividade do fornecedor quando comparada com o total de registros elegíveis
    CASE
        WHEN COUNT(*) < 100
            THEN 'Baixa'
        ELSE 'Adequada'
    END AS representatividade,

    -- Valor total das vendas elegíveis
    ROUND(
        SUM(e.vl_preco_total),
        2
    ) AS valor_total_vendas,

    -- Total de registros potencialmente atípicos
    COUNT(o.co_seq_bps) AS qtd_potencialmente_atipicos,

    -- Potencialmente atípicos abaixo do limite inferior
    SUM(
        CASE
            WHEN o.tipo_atipicidade = 'Abaixo do limite inferior'
            THEN 1
            ELSE 0
        END
    ) AS qtd_abaixo_limite,

    -- Potencialmente atípicos acima do limite superior
    SUM(
        CASE
            WHEN o.tipo_atipicidade = 'Acima do limite superior'
            THEN 1
            ELSE 0
        END
    ) AS qtd_acima_limite,

    -- Percentual total de registros potencialmente atípicos
    ROUND(
        100.0 * COUNT(o.co_seq_bps) / COUNT(*),
        2
    ) AS percentual_potencialmente_atipicos,

    -- Percentual abaixo do limite inferior
    ROUND(
        100.0 * SUM(
            CASE
                WHEN o.tipo_atipicidade = 'Abaixo do limite inferior'
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS percentual_abaixo_limite,

    -- Percentual acima do limite superior
    ROUND(
        100.0 * SUM(
            CASE
                WHEN o.tipo_atipicidade = 'Acima do limite superior'
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS percentual_acima_limite

FROM mvp_puc_engenhariadedados.gold.vw_registros_elegiveis e
LEFT JOIN mvp_puc_engenhariadedados.gold.vw_potenciais_outliers o
    ON e.co_seq_bps = o.co_seq_bps
GROUP BY e.cnpj_fornecedor, e.no_fornecedor
HAVING SUM(e.vl_preco_total) >= 1000000
ORDER BY percentual_potencialmente_atipicos DESC;